In [11]:
!pip uninstall -y transformers accelerate peft datasets

!pip install -q \
transformers==4.41.2 \
accelerate==0.30.1 \
datasets==2.19.1 \
peft==0.11.1

Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2
Found existing installation: accelerate 0.30.1
Uninstalling accelerate-0.30.1:
  Successfully uninstalled accelerate-0.30.1
Found existing installation: peft 0.11.1
Uninstalling peft-0.11.1:
  Successfully uninstalled peft-0.11.1
Found existing installation: datasets 2.19.1
Uninstalling datasets-2.19.1:
  Successfully uninstalled datasets-2.19.1


In [12]:
import transformers
import accelerate
import datasets
import peft

print(transformers.__version__)
print(accelerate.__version__)
print(datasets.__version__)
print(peft.__version__)

4.41.2
0.30.1
2.19.1
0.11.1


In [13]:
import torch
print(torch.cuda.is_available())
import pandas as pd
from transformers import AutoTokenizer
from dataset import LABELS, get_data
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

False


## Load datasets and replace [SEP] with a new line

In [14]:
data, thresholds = get_data()

x_train, y_train = data['train']
x_dev, y_dev = data["dev"]
x_test, y_test = data["test"]

x_train = x_train.apply(lambda x: x.replace(" [SEP] ", "\n"))
x_dev = x_dev.apply(lambda x: x.replace(" [SEP] ", "\n"))
x_test = x_test.apply(lambda x: x.replace(" [SEP] ", "\n"))

print (x_train[0])

Topic: We should abandon marriage
Stance: pro
Argument: "marriage" isn't keeping up with the times.  abandon the old thinking and bring something that incorporates all unions - not just those with a man and woman.


## Map labels to digits (e.g. low to 0)

In [ ]:
LABEL_TO_ID = {
    "low": 0,
    "medium": 1,
    "high": 2,
}

train_df = pd.DataFrame({
    "text": x_train,
    "label": y_train.map(LABEL_TO_ID)
})

dev_df = pd.DataFrame({
    "text": x_dev,
    "label": y_dev.map(LABEL_TO_ID)
})

test_df = pd.DataFrame({
    "text": x_test,
    "label": y_test.map(LABEL_TO_ID)
})

In [ ]:
print (train_df)

                                                    text  label
0      Topic: We should abandon marriage\nStance: pro...      1
1      Topic: We should adopt a multi-party system\nS...      1
2      Topic: Assisted suicide should be a criminal o...      0
3      Topic: We should abolish safe spaces\nStance: ...      0
4      Topic: We should ban naturopathy\nStance: pro\...      1
...                                                  ...    ...
20969  Topic: We should abolish zoos\nStance: pro\nAr...      2
20970  Topic: We should abolish zoos\nStance: pro\nAr...      1
20971  Topic: We should abolish zoos\nStance: pro\nAr...      2
20972  Topic: We should abolish zoos\nStance: con\nAr...      1
20973  Topic: We should abolish zoos\nStance: con\nAr...      1

[20974 rows x 2 columns]


## Initialize tokenizer

In [ ]:
#distilBERT
# tokenizer = AutoTokenizer.from_pretrained(
#     "distilbert-base-uncased"
# )

#roBERTa
tokenizer = AutoTokenizer.from_pretrained(
    "roberta-base"
)

print("tokenizer finished")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer finished


## Tokenize the input text and set the training arguments

In [ ]:
train_dataset = Dataset.from_pandas(train_df)
dev_dataset = Dataset.from_pandas(dev_df)
test_dataset = Dataset.from_pandas(test_df)

train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

dev_dataset = dev_dataset.map(
    tokenize,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True
)

#distilBERT
# model = AutoModelForSequenceClassification.from_pretrained(
#     "distilbert-base-uncased",
#     num_labels=3
# )

#roBERTa
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=3
)

#roBERTa
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.05,
    warmup_ratio=0.1,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1",
    greater_is_better=True,
    fp16=True,
    report_to="none",
    save_total_limit=2
)

#distilBERT
# training_args = TrainingArguments(
#     output_dir="./results",
#     evaluation_strategy="epoch",
#     save_strategy="epoch",
#     learning_rate=2e-5,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     num_train_epochs=3,
#     weight_decay=0.01,
#     logging_dir="./logs",
#     fp16=True
# )

Map:   0%|          | 0/20974 [00:00<?, ? examples/s]

Map:   0%|          | 0/3208 [00:00<?, ? examples/s]

Map:   0%|          | 0/6315 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


## Define metrics function

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(
            labels,
            predictions,
            average="macro"
        )
    }


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics
)

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


## Fine-tune and evaluate the model

In [ ]:
print ("start training")
trainer.train()
print ("finished training")
print ("start evaluation")
trainer.evaluate()
print ("finished evaluation")

start training


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.998000,0.993035,0.491584,0.482864
2,0.923600,1.054940,0.470075,0.444893
3,0.872200,1.054423,0.492207,0.475154
4,0.778400,1.187356,0.489401,0.476130
5,0.698000,1.282791,0.488778,0.473482


finished training
start evaluation


finished evaluation


In [ ]:
trainer.evaluate(test_dataset)

{'eval_loss': 1.0004905462265015,
 'eval_accuracy': 0.4943784639746635,
 'eval_macro_f1': 0.4883355153982216,
 'eval_runtime': 17.9047,
 'eval_samples_per_second': 352.7,
 'eval_steps_per_second': 44.122,
 'epoch': 5.0}

In [ ]:
model.save_pretrained("./argument_model-roberta")
tokenizer.save_pretrained("./argument_model-roberta")

('./argument_model-roberta/tokenizer_config.json',
 './argument_model-roberta/special_tokens_map.json',
 './argument_model-roberta/vocab.json',
 './argument_model-roberta/merges.txt',
 './argument_model-roberta/added_tokens.json',
 './argument_model-roberta/tokenizer.json')